# Late Fusion Multimodal Model

In [ ]:
import pandas as pd
import numpy as np
import torch
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    cohen_kappa_score,
    log_loss
)
from scipy.special import softmax
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, 
    DataCollatorWithPadding, TrainingArguments, Trainer
)
import evaluate

In [117]:
df = pd.read_csv("/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-alladults.csv") 

# clean texxt data
df["chiefcomplaint"] = df["chiefcomplaint"].astype(str).fillna("").str.strip()
df = df[df["chiefcomplaint"].str.len() > 0].copy()


df = df[df["acuity"].isin([1, 2, 3, 4, 5])].copy()
df["label"] = df["acuity"].astype(int) - 1  

# 60% Train, 20% Val, 20% Test
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
train_df, val_df = train_test_split(train_val_df, test_size=0.25, random_state=42, stratify=train_val_df["label"])

print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

Train size: 267489 | Val size: 89164 | Test size: 89164


## Tabular Baseline

In [119]:
cols_to_drop = ["chiefcomplaint", "acuity", "label", "race"] 


X_train_tab = train_df.drop(columns=cols_to_drop, errors='ignore')
y_train = train_df["label"].values

X_val_tab = val_df.drop(columns=cols_to_drop, errors='ignore')
y_val = val_df["label"].values

X_test_tab = test_df.drop(columns=cols_to_drop, errors='ignore')
y_test = test_df["label"].values

print("Training XGBoost on Tabular Data...")
xgb_model = xgb.XGBClassifier(  # these are the same parameters we used in our single-modality code
    n_estimators=500, 
    learning_rate=0.05, 
    objective='multi:softprob',
    num_class=5, 
    eval_metric='mlogloss', 
    early_stopping_rounds=25, 
    random_state=42)

xgb_model.fit(
    X_train_tab, y_train,
    eval_set=[(X_val_tab, y_val)],
    verbose=False)

# Extract Meta-Features (Probabilities)
print("Extracting Tabular Probabilities...")
tab_probs_train = xgb_model.predict_proba(X_train_tab)
tab_probs_val   = xgb_model.predict_proba(X_val_tab)
tab_probs_test  = xgb_model.predict_proba(X_test_tab)

Training XGBoost on Tabular Data...
Extracting Tabular Probabilities...


## Text Baseline

In [ ]:
model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["chiefcomplaint"], truncation=True, padding=True, max_length=32)

train_ds = Dataset.from_pandas(train_df[["chiefcomplaint", "label"]]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[["chiefcomplaint", "label"]]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[["chiefcomplaint", "label"]]).map(tokenize, batched=True)

# initialize model and trainer
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=5)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir="clinicalbert_triage", learning_rate=2e-5,
    per_device_train_batch_size=32, num_train_epochs=2,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, report_to="none")

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=data_collator)

print("Fine-tuning Bio_ClinicalBERT...")
trainer.train()

# logits -> softmax -> probabilities)
print("Extracting Text Probabilities...")
text_logits_train = trainer.predict(train_ds).predictions
text_logits_val   = trainer.predict(val_ds).predictions
text_logits_test  = trainer.predict(test_ds).predictions

text_probs_train = softmax(text_logits_train, axis=1)
text_probs_val   = softmax(text_logits_val, axis=1)
text_probs_test  = softmax(text_logits_test, axis=1)

## Multimodal (Late Fusion)

In [ ]:
def evaluate_metrics(y_true, y_pred, y_prob, label):
    print(f"\n--- {label} ---")
    ll = log_loss(y_true, y_prob)
    print("Log Loss (Error):", f"{ll:.4f}")

    print("Accuracy:", f"{accuracy_score(y_true, y_pred):.3f}")
    print("Balanced Accuracy:", f"{balanced_accuracy_score(y_true, y_pred):.3f}")
    print("Macro F1:", f"{f1_score(y_true, y_pred, average='macro'):.3f}")
    kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
    print("Cohen's Kappa (Quadratic):", f"{kappa:.3f}")

    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

# ------- This is an old version that doesn't compute log loss, but the pediatric tests use it so can comment/uncomment as needed ------------

# def evaluate_metrics(y_true, y_pred, label):
# 	print(f"\n--- {label} ---")
# 	print("Accuracy:", f"{accuracy_score(y_true, y_pred):.3f}")
# 	print("Balanced Accuracy:", f"{balanced_accuracy_score(y_true, y_pred):.3f}")
# 	print("Macro F1:", f"{f1_score(y_true, y_pred, average='macro'):.3f}")
# 	kappa = cohen_kappa_score(y_true, y_pred, weights='quadratic')
# 	print("Cohen's Kappa (Quadratic):", f"{kappa:.3f}")

# 	print("\nConfusion Matrix:")
# 	print(confusion_matrix(y_true, y_pred))
# 	print("\nClassification Report:")
# 	print(classification_report(y_true, y_pred))

def evaluate_asymmetric_dropout(drop_rate_text, drop_rate_tab, text_train, tab_train, y_train, text_test, tab_test, y_test):
    np.random.seed(42)
    n_samples = text_train.shape[0]
    
    dropped_text = text_train.copy()
    dropped_tab  = tab_train.copy()
    
    rand_rolls = np.random.rand(n_samples)
    
    # drop text
    drop_text_mask = rand_rolls < drop_rate_text
    dropped_text[drop_text_mask] = 0.0
    
    # drop tabular
    drop_tab_mask = (rand_rolls >= drop_rate_text) & (rand_rolls < (drop_rate_text + drop_rate_tab))  # prevents overlap so we don't drop both
    dropped_tab[drop_tab_mask] = 0.0

    X_train_meta = np.hstack((dropped_text, dropped_tab))

    # this is the model 
    meta_clf = LogisticRegression(max_iter=1000, random_state=42)
    meta_clf.fit(X_train_meta, y_train)

	# train (for training los loss/error)
    X_train_intact = np.hstack((text_train, tab_train))
    pred_train_intact = meta_clf.predict(X_train_intact)
    prob_train_intact = meta_clf.predict_proba(X_train_intact)

    X_test_both = np.hstack((text_test, tab_test))
    pred_both = meta_clf.predict(X_test_both)
    prob_both = meta_clf.predict_proba(X_test_both)
    
    # text only
    X_test_no_tab = np.hstack((text_test, np.zeros_like(tab_test)))
    pred_no_tab = meta_clf.predict(X_test_no_tab)
    prob_no_tab = meta_clf.predict_proba(X_test_no_tab)
    
    # tabular only
    X_test_no_text = np.hstack((np.zeros_like(text_test), tab_test))
    pred_no_text = meta_clf.predict(X_test_no_text)
    prob_no_text = meta_clf.predict_proba(X_test_no_text)
    

    print(f"\n======== Dropout Training Rates ========")
    print(f"Text: {drop_rate_text*100:.0f}% | Tabular: {drop_rate_tab*100:.0f}%")

    evaluate_metrics(y_train, pred_train_intact, prob_train_intact, "Train Set (both modalities)")
    evaluate_metrics(y_test, pred_both, prob_both, "Test Set (both modalities)")
    evaluate_metrics(y_test, pred_no_tab, prob_no_tab, "Test Set - Text Only (tabular dropped)")
    evaluate_metrics(y_test, pred_no_text, prob_no_text, "Test Set - Tabular Only (text dropped)")
    
    return meta_clf

#### Symmetric Dropout (Adults)

In [ ]:
dropout_pairs = [
    (0.00, 0.00), # Baseline: No dropout
	(0.05, 0.05),
	(0.10, 0.10),
    (0.15, 0.15), # Balanced dropout (30% total)
	(0.20, 0.20),
	(0.30, 0.30),
	(0.40, 0.40),
	(0.50, 0.50),
	(0.60, 0.60),
]

meta_X_val_text = text_probs_val
meta_X_val_tab  = tab_probs_val
meta_y_val      = y_val

meta_X_test_text = text_probs_test
meta_X_test_tab  = tab_probs_test
meta_y_test      = y_test

trained_meta_models = {}

print("\n" + "="*50)
print(" EVALUATING ROBUSTNESS ON ADULT COHORT")
print("="*50)
for drop_text, drop_tab in dropout_pairs:
    clf = evaluate_asymmetric_dropout(
        drop_text, drop_tab,
        meta_X_val_text, meta_X_val_tab, meta_y_val,
        meta_X_test_text, meta_X_test_tab, meta_y_test
    )
    trained_meta_models[(drop_text, drop_tab)] = clf


 EVALUATING ROBUSTNESS ON ADULT COHORT

======== Dropout Training Rates ========
Text: 0% | Tabular: 0%

--- Train Set (Both Modalities Intact) ---
Log Loss (Error): 0.7209
Accuracy: 0.697
Balanced Accuracy: 0.468
Macro F1: 0.494
Cohen's Kappa (Quadratic): 0.630

Confusion Matrix:
[[ 2777  1542   468    19     2]
 [  843 17824  9860   125     1]
 [  140  7098 39226  1468     2]
 [   16    88  4894  2320     6]
 [   11    17   168   247     2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.58      0.65      4808
           1       0.67      0.62      0.65     28653
           2       0.72      0.82      0.77     47934
           3       0.56      0.32      0.40      7324
           4       0.15      0.00      0.01       445

    accuracy                           0.70     89164
   macro avg       0.57      0.47      0.49     89164
weighted avg       0.69      0.70      0.69     89164


--- Test Set (Both Modalities Intact)

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


======== Dropout Training Rates ========
Text: 5% | Tabular: 5%

--- Train Set (Both Modalities Intact) ---
Log Loss (Error): 0.7208
Accuracy: 0.697
Balanced Accuracy: 0.469
Macro F1: 0.495
Cohen's Kappa (Quadratic): 0.631

Confusion Matrix:
[[ 2776  1546   465    19     2]
 [  842 17864  9816   131     0]
 [  138  7122 39175  1498     1]
 [   19    89  4860  2355     1]
 [   11    18   168   246     2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.58      0.65      4808
           1       0.67      0.62      0.65     28653
           2       0.72      0.82      0.77     47934
           3       0.55      0.32      0.41      7324
           4       0.33      0.00      0.01       445

    accuracy                           0.70     89164
   macro avg       0.60      0.47      0.49     89164
weighted avg       0.69      0.70      0.69     89164


--- Test Set (Both Modalities Intact) ---
Log Loss (Error): 0.7237
Accuracy: 

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


======== Dropout Training Rates ========
Text: 10% | Tabular: 10%

--- Train Set (Both Modalities Intact) ---
Log Loss (Error): 0.7210
Accuracy: 0.698
Balanced Accuracy: 0.469
Macro F1: 0.495
Cohen's Kappa (Quadratic): 0.631

Confusion Matrix:
[[ 2772  1555   461    18     2]
 [  840 17882  9799   132     0]
 [  136  7116 39187  1493     2]
 [   19    89  4866  2349     1]
 [   11    18   170   244     2]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.58      0.65      4808
           1       0.67      0.62      0.65     28653
           2       0.72      0.82      0.77     47934
           3       0.55      0.32      0.41      7324
           4       0.29      0.00      0.01       445

    accuracy                           0.70     89164
   macro avg       0.59      0.47      0.49     89164
weighted avg       0.69      0.70      0.69     89164


--- Test Set (Both Modalities Intact) ---
Log Loss (Error): 0.7240
Accuracy

...

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


======== Dropout Training Rates ========
Text: 50% | Tabular: 50%

--- Train Set (Both Modalities Intact) ---
Log Loss (Error): 0.7816
Accuracy: 0.687
Balanced Accuracy: 0.406
Macro F1: 0.441
Cohen's Kappa (Quadratic): 0.573

Confusion Matrix:
[[ 2132  1890   777     9     0]
 [  491 15278 12833    51     0]
 [   47  4726 42654   507     0]
 [    3    61  6066  1194     0]
 [    7    14   253   171     0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.44      0.57      4808
           1       0.70      0.53      0.60     28653
           2       0.68      0.89      0.77     47934
           3       0.62      0.16      0.26      7324
           4       0.00      0.00      0.00       445

    accuracy                           0.69     89164
   macro avg       0.56      0.41      0.44     89164
weighted avg       0.68      0.69      0.66     89164


--- Test Set (Both Modalities Intact) ---
Log Loss (Error): 0.7852
Accuracy

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}


======== Dropout Training Rates ========
Text: 60% | Tabular: 60%

--- Train Set (Both Modalities Intact) ---
Log Loss (Error): 0.7817
Accuracy: 0.688
Balanced Accuracy: 0.405
Macro F1: 0.438
Cohen's Kappa (Quadratic): 0.574

Confusion Matrix:
[[ 2150  1914   735     9     0]
 [  492 15617 12499    45     0]
 [   49  4964 42495   426     0]
 [    3    67  6182  1072     0]
 [    9    15   266   155     0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.45      0.57      4808
           1       0.69      0.55      0.61     28653
           2       0.68      0.89      0.77     47934
           3       0.63      0.15      0.24      7324
           4       0.00      0.00      0.00       445

    accuracy                           0.69     89164
   macro avg       0.56      0.41      0.44     89164
weighted avg       0.68      0.69      0.66     89164


--- Test Set (Both Modalities Intact) ---
Log Loss (Error): 0.7854
Accuracy

/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

#### Asymmetric Dropout (Adults)

In [ ]:
rates = [round(x, 1) for x in np.arange(0.1, 0.9, 0.1)]  # round() to prevent floating point precision issues

grid_results = []
trained_grid_models = {}

print("\n" + "="*60)
print(" ASYMMETRIC DROPOUT GRID SEARCH (0.1 to 0.8)")
print("="*60)

for drop_text in rates:
    for drop_tab in rates:
        
        clf = evaluate_asymmetric_dropout(
            drop_text, drop_tab,
            meta_X_val_text, meta_X_val_tab, meta_y_val,
            meta_X_test_text, meta_X_test_tab, meta_y_test
        )
        
        # store the model
        trained_grid_models[(drop_text, drop_tab)] = clf
        
        # calculate accuracy and QWK
        X_test_combined = np.hstack((meta_X_test_text, meta_X_test_tab))
        y_pred = clf.predict(X_test_combined)
        
        acc = accuracy_score(meta_y_test, y_pred)
        qwk = cohen_kappa_score(meta_y_test, y_pred, weights='quadratic')
        
        grid_results.append({
            'Drop_Text': drop_text,
            'Drop_Tab': drop_tab,
            'Accuracy': acc,
            'QWK': qwk
        })
        
        print(f"Text Drop: {drop_text:.1f} | Tab Drop: {drop_tab:.1f}  -->  Acc: {acc:.3f} | QWK: {qwk:.3f}")

results_df = pd.DataFrame(grid_results)
save_path = '/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/asymmetric.csv'
results_df.to_csv(save_path, index=False)

print(f"Data successfully exported to: {save_path}")
display(results_df.head(10)) 

### Pediatric Cohort Test

In [ ]:
print("\n" + "="*50)
print(" PREPARING PEDIATRIC DATASET (EXTERNAL VALIDATION)")
print("="*50)

peds_path = '/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/tyrm-pedsNHAMCS.csv'
peds_df = pd.read_csv(peds_path)

peds_df["chiefcomplaint"] = peds_df["chiefcomplaint"].fillna("").astype(str).str.strip()
peds_df = peds_df[peds_df["chiefcomplaint"].str.len() > 0].copy()  # this should have already been done in pre-processing

peds_df = peds_df[peds_df["acuity"].isin([1, 2, 3, 4, 5])].copy()
peds_df["label"] = peds_df["acuity"].astype(int) - 1

meta_y_peds = peds_df["label"].values

print(f"Pediatric cohort size after filtering: {len(peds_df)}")

# process tabular data
X_peds_tab = peds_df.drop(columns=cols_to_drop, errors="ignore")


print("Extracting Peds Tabular Probabilities (XGBoost)...")
meta_X_peds_tab = xgb_model.predict_proba(X_peds_tab)

# process text data
peds_ds = Dataset.from_pandas(peds_df[["chiefcomplaint", "label"]])
peds_ds = peds_ds.map(tokenize, batched=True, remove_columns=["chiefcomplaint"])

print("Extracting Peds Text Probabilities (Bio_ClinicalBERT)...")
peds_logits = trainer.predict(peds_ds).predictions
meta_X_peds_text = softmax(peds_logits, axis=1)


 PREPARING PEDIATRIC DATASET (EXTERNAL VALIDATION)
Pediatric cohort size after filtering: 5581
Extracting Peds Tabular Probabilities (XGBoost)...


Map:   0%|          | 0/5581 [00:00<?, ? examples/s]

Extracting Peds Text Probabilities (Bio_ClinicalBERT)...


/opt/miniconda3/envs/triage229/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

#### Symmetric Dropout (Peds Cohort)

In [ ]:
print("\n" + "="*50)
print(" EVALUATING ROBUSTNESS ON PEDIATRIC COHORT")
print("="*50)

peds_results = []

for (drop_text, drop_tab), meta_clf in trained_meta_models.items():

    # both modalities
    X_peds_both = np.hstack((meta_X_peds_text, meta_X_peds_tab))
    pred_both = meta_clf.predict(X_peds_both)

    # text only
    X_peds_no_tab = np.hstack((meta_X_peds_text, np.zeros_like(meta_X_peds_tab)))
    pred_no_tab = meta_clf.predict(X_peds_no_tab)

    # tabular only
    X_peds_no_text = np.hstack((np.zeros_like(meta_X_peds_text), meta_X_peds_tab))
    pred_no_text = meta_clf.predict(X_peds_no_text)

    print(f"\n{'='*60}")
    print(f"Dropout Training Rates -> Text: {drop_text*100:.0f}%, Tab: {drop_tab*100:.0f}%")
    print(f"{'='*60}")

	# MAKE SURE THE CORRECT VERSION OF EVALUATE_METRICS IS UNCOMMENTED
    evaluate_metrics(meta_y_peds, pred_both, "Peds: Both Modalities Intact")
    evaluate_metrics(meta_y_peds, pred_no_tab, "Peds: Text Only (Tabular Dropped)")
    evaluate_metrics(meta_y_peds, pred_no_text, "Peds: Tabular Only (Text Dropped)")


#### Asymmetric Dropout (Peds Cohort)

In [ ]:
print("\n" + "="*60)
print(" EVALUATING ROBUSTNESS ON PEDIATRIC COHORT")
print("="*60)

peds_results = []

# loop through all the models we trained in the previous grid search
for (drop_text, drop_tab), meta_clf in trained_grid_models.items():

    # both
    X_peds_both = np.hstack((meta_X_peds_text, meta_X_peds_tab))
    pred_both = meta_clf.predict(X_peds_both)

    # text only
    X_peds_no_tab = np.hstack((meta_X_peds_text, np.zeros_like(meta_X_peds_tab)))
    pred_no_tab = meta_clf.predict(X_peds_no_tab)

    # tabular only
    X_peds_no_text = np.hstack((np.zeros_like(meta_X_peds_text), meta_X_peds_tab))
    pred_no_text = meta_clf.predict(X_peds_no_text)

    # metrics for all three conditions
    qwk_both = cohen_kappa_score(meta_y_peds, pred_both, weights='quadratic')
    acc_both = accuracy_score(meta_y_peds, pred_both)
    
    qwk_no_tab = cohen_kappa_score(meta_y_peds, pred_no_tab, weights='quadratic')
    acc_no_tab = accuracy_score(meta_y_peds, pred_no_tab)
    
    qwk_no_text = cohen_kappa_score(meta_y_peds, pred_no_text, weights='quadratic')
    acc_no_text = accuracy_score(meta_y_peds, pred_no_text)

    peds_results.append({
        'Drop_Text': drop_text,
        'Drop_Tab': drop_tab,
        'QWK_Both': qwk_both,
        'Acc_Both': acc_both,
        'QWK_No_Tab': qwk_no_tab,
        'Acc_No_Tab': acc_no_tab,
        'QWK_No_Text': qwk_no_text,
        'Acc_No_Text': acc_no_text
    })
    print(f"Train Drop [Text:{drop_text:.1f}, Tab:{drop_tab:.1f}] -> Peds QWK (Both Intact): {qwk_both:.3f}")

peds_df = pd.DataFrame(peds_results)

save_path = '/Users/Tyler/Documents/Stanford Classes/CS229/CS229Final/Dataframes/peds_asymmetric.csv'
peds_df.to_csv(save_path, index=False)

print(f"\nPediatric evaluation complete. Data successfully exported to:\n{save_path}")

#### Age Stratification Analysis

In [ ]:
print("\n" + "="*50)
print(" ERROR ANALYSIS: AGE STRATIFICATION (40% DROPOUT)")
print("="*50)

best_clf = trained_meta_models[(0.40, 0.40)]  # best performing symmetric dropout model

# define standard clinical pediatric age brackets
age_brackets = {
    "Infants (0-1 years)": (0, 1),
    "Toddlers/Preschool (2-5 years)": (2, 5),
    "School Age (6-12 years)": (6, 12),
    "Adolescents (13-17 years)": (13, 17)
}

age_col = 'age_at_visit' 

for bracket_name, (age_min, age_max) in age_brackets.items():
    mask = (peds_df[age_col] >= age_min) & (peds_df[age_col] <= age_max)
    
    n_patients = mask.sum()
    if n_patients == 0:
        print(f"\n--- {bracket_name} ---")
        print("No patients found in this age range.")
        continue
        
    # apply mask
    y_true_bracket = meta_y_peds[mask]
    X_text_bracket = meta_X_peds_text[mask]
    X_tab_bracket  = meta_X_peds_tab[mask]
    
    # both
    X_perfect = np.hstack((X_text_bracket, X_tab_bracket))
    acc_perfect = accuracy_score(y_true_bracket, best_clf.predict(X_perfect))
    
    # no tabular
    X_no_tab = np.hstack((X_text_bracket, np.zeros_like(X_tab_bracket)))
    acc_no_tab = accuracy_score(y_true_bracket, best_clf.predict(X_no_tab))
    
    # no text
    X_no_text = np.hstack((np.zeros_like(X_text_bracket), X_tab_bracket))
    acc_no_text = accuracy_score(y_true_bracket, best_clf.predict(X_no_text))
    
    print(f"\n--- {bracket_name} (n={n_patients}) ---")
    print(f"Accuracy (Both Intact):  {acc_perfect:.3f}")
    print(f"Accuracy (No Tabular):   {acc_no_tab:.3f}")
    print(f"Accuracy (No Text):      {acc_no_text:.3f}")